# 02 — Data cleaning

The code for each step is in its own file in [`cleaning/`](cleaning/). This notebook runs the steps one by
one so you can see what each one does. Open the step file to read the code and the reasons behind it.

| Step | File | What it does |
|---|---|---|
| 1 | `cleaning/step1_types.py` | Parse times; True/False → 1/0 |
| 2 | `cleaning/step2_duplicates.py` | Remove exact copies and accidents reported twice |
| 3 | `cleaning/step3_columns.py` | Drop leakage, constant, redundant, artefact and rare columns |
| 4 | `cleaning/step4_missing.py` | Missing values |
| 5 | `cleaning/step5_outliers.py` | Impossible values → NaN |
| 6 | `cleaning/step6_weather.py` | Weather_Condition (108 labels) → Weather_Group (8) |
| 7 | `cleaning/step7_save.py` | Checks, save `US_Accidents_cleaned.csv`, write `reports/cleaning_log.md` |

Run everything at once from a terminal: `python -m cleaning`.
The full explanation with charts is in `reports/Data_Cleaning_Report.pdf`.
The original CSV is only read, never written.

In [1]:
import pandas as pd

from cleaning import (step1_types, step2_duplicates, step3_columns, step4_missing,
                      step5_outliers, step6_weather, step7_save)
from cleaning.common import RAW_PATH, CleaningLog, md5

raw_md5 = md5(RAW_PATH)
df = pd.read_csv(RAW_PATH)
log = CleaningLog()
df.shape

(500000, 46)

## Step 1 — Types (`cleaning/step1_types.py`)

In [2]:
df = step1_types.run(df, log)
df[step1_types.TIME_COLS + step1_types.ROAD_COLS[:2]].dtypes

1. Types                           500,000 x 46 -> 500,000 x 46  times parsed to datetime; road flags True/False -> 1/0


Start_Time           datetime64[ns]
End_Time             datetime64[ns]
Weather_Timestamp    datetime64[ns]
Amenity                       int64
Bump                          int64
dtype: object

## Step 2 — Duplicates (`cleaning/step2_duplicates.py`)

In [3]:
df = step2_duplicates.run(df, log)
{k: log.notes[k] for k in ["repeated_event_rows", "repeated_event_conflicting_severity"]}

2a. Exact duplicates               500,000 x 46 -> 499,497 x 46  rows identical on every column except ID


2b. Same accident reported twice   499,497 x 46 -> 496,856 x 46  same Start_Time + Start_Lat + Start_Lng; kept the highest Severity


{'repeated_event_rows': 5125, 'repeated_event_conflicting_severity': 260}

## Step 3 — Column audit (`cleaning/step3_columns.py`)

In [4]:
df = step3_columns.run(df, log)
pd.DataFrame({"category": step3_columns.DROP_CATEGORY, "reason": step3_columns.DROP_REASONS})

3. Column audit                    496,856 x 46 -> 496,856 x 22  dropped 24 columns


,category,reason
End_Time,Leakage,Known only once the road is cleared. Duration ...
End_Lat,Leakage,"End of the affected stretch of road, measured ..."
End_Lng,Leakage,Same as End_Lat
Distance(mi),Leakage,"Length of road affected, part of how Severity ..."
Description,Leakage,"Free text written afterwards ('Road closed', '..."
ID,No information,"Identifier, different on every row"
Country,No information,Constant: always 'US'
Turning_Loop,No information,Constant: always False
Wind_Chill(F),Redundant,Correlation 0.994 with Temperature (77% identi...
Civil_Twilight,Redundant,Equal to Sunrise_Sunset in 95% of rows


## Step 4 — Missing values (`cleaning/step4_missing.py`)

In [5]:
df = step4_missing.run(df, log)
df.isna().mean().mul(100).round(2).loc[lambda s: s > 0].rename("% missing (left for the model pipeline)")

4a. Missing City                   496,856 x 22 -> 496,837 x 22  dropped rows with no City


4b. Fill / tidy                    496,837 x 22 -> 496,837 x 22  Sunrise_Sunset NaN -> 'Unknown'; Street stripped of spaces

Street               0.14
Temperature(F)       2.09
Humidity(%)          2.22
Pressure(in)         1.78
Visibility(mi)       2.26
Wind_Speed(mph)      7.43
Weather_Condition    2.22
Name: % missing (left for the model pipeline), dtype: float64

## Step 5 — Impossible values (`cleaning/step5_outliers.py`)

In [6]:
df = step5_outliers.run(df, log)
log.notes["impossible_values"]

5. Impossible values               496,837 x 22 -> 496,837 x 22  set to NaN: Temperature(F) 4, Wind_Speed(mph) 7, Visibility(mi) 3, Pressure(in) 3, Humidity(%) 0


{'Temperature(F)': 4,
 'Wind_Speed(mph)': 7,
 'Visibility(mi)': 3,
 'Pressure(in)': 3,
 'Humidity(%)': 0}

## Step 6 — Weather groups (`cleaning/step6_weather.py`)

In [7]:
df = step6_weather.run(df, log)
df["Weather_Group"].value_counts()

6. Weather grouping                496,837 x 22 -> 496,837 x 22  Weather_Condition (108 labels) -> Weather_Group (8 groups)

Weather_Group
Clear                      216996
Cloudy                     200408
Rain                        30691
Fog/Haze/Smoke              13057
Snow/Ice                    11387
Unknown                     11222
Heavy rain/Thunderstorm      7915
Windy                        5161
Name: count, dtype: int64

## Step 7 — Check and save (`cleaning/step7_save.py`)

In [8]:
df = step7_save.run(df, log, raw_md5)
log.to_frame()

7. Save                            496,837 x 22 -> 496,837 x 22  wrote US_Accidents_cleaned.csv; original CSV unchanged


,step,detail,rows_before,rows_after,cols_before,cols_after
0,1. Types,times parsed to datetime; road flags True/Fals...,500000,500000,46,46
1,2a. Exact duplicates,rows identical on every column except ID,500000,499497,46,46
2,2b. Same accident reported twice,same Start_Time + Start_Lat + Start_Lng; kept ...,499497,496856,46,46
3,3. Column audit,dropped 24 columns,496856,496856,46,22
4,4a. Missing City,dropped rows with no City,496856,496837,22,22
5,4b. Fill / tidy,Sunrise_Sunset NaN -> 'Unknown'; Street stripp...,496837,496837,22,22
6,5. Impossible values,"set to NaN: Temperature(F) 4, Wind_Speed(mph) ...",496837,496837,22,22
7,6. Weather grouping,Weather_Condition (108 labels) -> Weather_Grou...,496837,496837,22,22
8,7. Save,wrote US_Accidents_cleaned.csv; original CSV u...,496837,496837,22,22


## Warning for modelling: the Severity labels drift by Source and year

Source1 stops using Severity 3 from 2021, and Severity 1 almost only appears in 2020 (Source1) and
2022 (Source2/3). Keep `Source` and the year as features, and expect the time-based test to drop
partly because of this. See section 10 of the PDF report.

In [9]:
pd.crosstab([df["Start_Time"].dt.year.rename("year"), df["Source"]], df["Severity"])

Severity         1      2      3     4
year Source                           
2016 Source1     0   5970   1489   867
     Source2    19  11546   6656    12
     Source3     0     51     43     0
2017 Source1     0   7387   2102  1531
     Source2    17  22197  12774    18
     Source3     0    266    214     1
2018 Source1     0   7163   1995  1583
     Source2    12  29544  16499    28
     Source3     0    369    366     0
2019 Source1     0  13357   2033  1605
     Source2    16  30310  13049   229
     Source3     0    791    430     0
2020 Source1  1771  38867   2970  1710
     Source2    14  19809   8650   288
     Source3     2    997    405     0
2021 Source1     0  69803      0  1682
     Source2     6  18355   9451   254
     Source3     1   1020    240     1
2022 Source1     0  94345      0  2638
     Source2  2119   7175   4851   101
     Source3   276    605    251     3
2023 Source1     0  15164      0   474